# JCP Fall 26 — Week 7: SQL

This notebook goes with the Week 7 SQL slides. No installs: Python already ships with **SQLite**, a real database that lives in one file. We send it SQL as a string and get a pandas DataFrame back.

1. **Predict**: read the cell and write your guess in the `# Guess:` comment *before* you run it.
2. **Fix**: the cell is broken on purpose. Read the error (or the wrong output), find the bug, fix it.
3. **Tinker**: change one value, rerun, and notice what changes.
4. **Your Turn**: replace every `...` with your own SQL.

Examples use a cafe database: **`customers`** and **`orders`**. Your Turn cells use a music streaming app: **`users`**, **`songs`**, and **`plays`**. Same skills, different tables, and there are a lot of Your Turn questions, so pace yourself.

Keep each cell short. If you're stuck for more than 2 minutes, ask your partner, then ask us.

---
## 0. Setup

Run these cells once. The first one builds `week7.db` in the same folder as this notebook (you don't need to read it).

In [40]:
import sqlite3
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# ---------- cafe database (examples) ----------
customers = pd.DataFrame({
    "customer_id":   [1, 2, 3, 4, 5, 6, 7, 8],
    "customer_name": ["Ava", "Ben", "Chloe", "Dev", "Emma", "Farah", "Gabe", "Hana"],
    "city":          ["Berkeley", "Oakland", "Berkeley", "San Francisco", "Oakland", "Berkeley", "San Francisco", "Berkeley"],
    "nickname":      [None, "Benny", None, "D", None, None, "G", None],
})
menu = {"Latte": 5.25, "Mocha": 5.75, "Matcha": 6.00, "Cold Brew": 4.75, "Tea": 3.50, "Chai Latte": 5.50}
order_rows = []
for order_id in range(1, 41):
    cid = int(rng.choice([1, 1, 2, 3, 3, 3, 4, 5, 6, 6, 7, 9]))      # customer 8 never orders, customer 9 doesn't exist
    drink = str(rng.choice(list(menu)))
    size = rng.choice(["small", "medium", "large", None], p=[0.3, 0.4, 0.2, 0.1])
    price = round(menu[drink] + {"small": -0.5, "medium": 0, "large": 0.75, None: 0}[size], 2)
    day = int(rng.integers(1, 31))
    order_rows.append([order_id, cid, drink, size, price, f"2026-10-{day:02d}"])
orders = pd.DataFrame(order_rows, columns=["order_id", "customer_id", "drink", "size", "price", "order_date"])

# ---------- music streaming database (Your Turn) ----------
users = pd.DataFrame({
    "user_id": range(1, 26),
    "username": ["lofi_luna", "beatsbyben", "kenji", "mango", "vinyl_vic", "zara9", "night_owl", "dj_dani",
                 "ritz", "solar_sam", "pixel_pia", "echo", "moonlit_mo", "tuneup_tia", "rhythm_raj", "aria",
                 "bassline_bo", "cadence", "indie_ines", "loop_leo", "melody_mia", "noah", "groove_gus", "harmony_h", "kai_k"],
    "country": rng.choice(["US", "Canada", "Mexico", "UK", "India"], size=25, p=[0.4, 0.15, 0.15, 0.15, 0.15]),
    "plan": rng.choice(["free", "premium", "student"], size=25, p=[0.4, 0.3, 0.3]),
    "signup_date": [f"2025-{int(m):02d}-{int(d):02d}" for m, d in zip(rng.integers(1, 13, 25), rng.integers(1, 29, 25))],
    "referred_by": [None, None, 1, None, 1, 2, None, 4, 1, None, 5, None, 2, None, 4, 9, None, 4, None, 12,
                    None, 3, None, 9, None],
})
genre_artists = {
    "Pop": ["Neon Harbor", "Sugar Static"], "Indie": ["Paper Kites Club", "Little Lanterns"],
    "Hip-Hop": ["MC Parallax", "Lyric Theory"], "Lo-fi": ["Rainy Window"], "Rock": ["Cinder Avenue"],
    "R&B": ["Velvet Hour", "Juno Rae"],
}
titles = ["Love on Mute", "Glass Skyline", "Midnight Bus", "Paper Planes Again", "Static Love", "Golden Hour Freestyle",
          "Coffee Rings", "Northbound", "Love Letters in Pencil", "Satellite Hearts", "Rooftop Summer", "Slow Fade",
          "Fever Dream", "Late Checkout", "Lovesick Algorithm", "Polaroid", "Second Wind", "Cloud Nine Remix",
          "Backseat Radio", "Low Tide", "Wildfire", "Echo Chamber", "City of Lights", "Daydream Arcade",
          "Bittersweet", "Heatwave", "Snow Globe", "Overtime", "Afterglow", "Window Seat"]
song_rows = []
for i, title in enumerate(titles, start=1):
    genre = list(genre_artists)[i % 6]
    artist = genre_artists[genre][i % len(genre_artists[genre])]
    if genre == "Rock":
        duration = int(rng.integers(220, 360))
    elif genre == "Lo-fi":
        duration = int(rng.integers(95, 180))
    else:
        duration = int(rng.integers(140, 300))
    song_rows.append([i, title, artist, genre, duration, int(rng.integers(2008, 2027))])
songs = pd.DataFrame(song_rows, columns=["song_id", "title", "artist", "genre", "duration_sec", "release_year"])

song_weights = rng.gamma(2.0, 1.0, size=len(songs))
song_weights[[26, 28]] = 0                       # two songs nobody plays
favorite_genre = {"US": "Hip-Hop", "Canada": "Indie", "Mexico": "Pop", "UK": "Rock", "India": "Lo-fi"}
skip_rate = {"Pop": 0.25, "Indie": 0.2, "Hip-Hop": 0.3, "Lo-fi": 0.1, "Rock": 0.35, "R&B": 0.2}
play_rows = []
play_id = 1
for uid in users["user_id"]:
    if uid in (12, 23):                          # two users who signed up but never listened
        continue
    n_plays = int(rng.integers(5, 60))
    country = users.loc[users["user_id"] == uid, "country"].iloc[0]
    w = song_weights * np.where(songs["genre"] == favorite_genre[country], 4.0, 1.0)
    w = w / w.sum()
    for _ in range(n_plays):
        s = songs.iloc[rng.choice(len(songs), p=w)]
        skipped = rng.random() < skip_rate[s["genre"]]
        seconds = int(rng.integers(3, 30)) if skipped else int(s["duration_sec"])
        play_rows.append([play_id, int(uid), int(s["song_id"]), f"2026-{int(rng.integers(8, 11)):02d}-{int(rng.integers(1, 29)):02d}", seconds])
        play_id += 1
plays = pd.DataFrame(play_rows, columns=["play_id", "user_id", "song_id", "play_date", "seconds_listened"])

conn = sqlite3.connect("week7.db")
for name, df in [("customers", customers), ("orders", orders), ("users", users), ("songs", songs), ("plays", plays)]:
    df.to_sql(name, conn, if_exists="replace", index=False)
print("database ready")

database ready


This helper is the only Python you need today: it sends a SQL string to the database and returns a DataFrame.

In [41]:
def sql(query):
    return pd.read_sql(query, conn)

sql("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,customers
1,orders
2,users
3,songs
4,plays


In [42]:
# FULL OUTER JOIN needs SQLite 3.39 or newer. If yours is older, you can skip that one cell later.
sqlite3.sqlite_version

'3.53.4'

**Example:** peek at each table. The triple quotes `"""` let a query span several lines.

In [43]:
sql("""
SELECT *
FROM customers
""")

,customer_id,customer_name,city,nickname
0,1,Ava,Berkeley,NaN
1,2,Ben,Oakland,Benny
2,3,Chloe,Berkeley,NaN
3,4,Dev,San Francisco,D
4,5,Emma,Oakland,NaN
5,6,Farah,Berkeley,NaN
6,7,Gabe,San Francisco,G
7,8,Hana,Berkeley,NaN


In [44]:
sql("""
SELECT *
FROM orders
LIMIT 5
""")

,order_id,customer_id,drink,size,price,order_date
0,1,9,Cold Brew,large,5.50,2026-10-18
1,2,6,Chai Latte,medium,5.50,2026-10-07
2,3,3,Chai Latte,small,5.00,2026-10-15
3,4,6,Latte,medium,5.25,2026-10-24
4,5,6,Mocha,small,5.25,2026-10-22


**Your Turn:** the music app has **three** tables: `users`, `songs`, and `plays` (one row per time someone played a song). Look at the first 5 rows of each.

In [45]:
sql("""
SELECT *
FROM users
LIMIT 5
""")

,user_id,username,country,plan,signup_date,referred_by
0,1,lofi_luna,India,free,2025-09-24,NaN
1,2,beatsbyben,Mexico,free,2025-04-17,NaN
2,3,kenji,Canada,free,2025-10-04,1.0
3,4,mango,Canada,free,2025-10-21,NaN
4,5,vinyl_vic,India,free,2025-09-17,1.0


In [46]:
sql("""
SELECT *
FROM songs
LIMIT 5
""")

,song_id,title,artist,genre,duration_sec,release_year
0,1,Love on Mute,Little Lanterns,Indie,248,2015
1,2,Glass Skyline,MC Parallax,Hip-Hop,140,2022
2,3,Midnight Bus,Rainy Window,Lo-fi,154,2023
3,4,Paper Planes Again,Cinder Avenue,Rock,345,2021
4,5,Static Love,Juno Rae,R&B,274,2010


In [47]:
sql("""
SELECT *
FROM plays
LIMIT 5
""")

,play_id,user_id,song_id,play_date,seconds_listened
0,1,1,23,2026-08-24,212
1,2,1,15,2026-08-25,155
2,3,1,20,2026-09-21,13
3,4,1,21,2026-10-19,104
4,5,1,12,2026-09-27,9


*Which column links `plays` to `users`? Which links `plays` to `songs`? What do you think `referred_by` means?*

`plays.user_id` links to `users.user_id`, and `plays.song_id` links to `songs.song_id`. `referred_by` holds the `user_id` of the user who invited this user to the app (`NULL` if nobody referred them).

---
## 1. SELECT, WHERE, ORDER BY, LIMIT

**Example:** build one query a clause at a time. Run each cell and watch the result change. We want to return the customer_id, drink, and price of the three most expensive orders that cost more than $5, sorted from highest price to lowest

In [48]:
sql("""
SELECT customer_id, drink, price
FROM orders
""")

,customer_id,drink,price
0,9,Cold Brew,5.50
1,6,Chai Latte,5.50
2,3,Chai Latte,5.00
3,6,Latte,5.25
4,6,Mocha,5.25
5,3,Chai Latte,5.50
6,4,Cold Brew,4.75
7,6,Tea,3.50
8,3,Mocha,5.25
9,5,Latte,4.75


In [49]:
sql("""
SELECT customer_id, drink, price
FROM orders
WHERE price > 5
""")

,customer_id,drink,price
0,9,Cold Brew,5.50
1,6,Chai Latte,5.50
2,6,Latte,5.25
3,6,Mocha,5.25
4,3,Chai Latte,5.50
5,3,Mocha,5.25
6,3,Matcha,5.50
7,1,Latte,5.25
8,7,Mocha,5.75
9,4,Mocha,6.50


In [50]:
sql("""
SELECT customer_id, drink, price
FROM orders
WHERE price > 5
ORDER BY price DESC
""")

,customer_id,drink,price
0,4,Mocha,6.50
1,6,Matcha,6.00
2,2,Latte,6.00
3,4,Latte,6.00
4,6,Latte,6.00
5,7,Mocha,5.75
6,3,Mocha,5.75
7,4,Mocha,5.75
8,9,Cold Brew,5.50
9,6,Chai Latte,5.50


In [51]:
sql("""
SELECT customer_id, drink, price
FROM orders
WHERE price > 5
ORDER BY price DESC
LIMIT 3
""")

,customer_id,drink,price
0,4,Mocha,6.5
1,6,Matcha,6.0
2,2,Latte,6.0


Same thing in pandas: `orders[orders["price"] > 5].sort_values("price", ascending=False).head(3)[["customer_id", "drink", "price"]]`.


SQL WHERE

In [52]:
sql("""
SELECT drink, size, price
FROM orders
WHERE drink IN ('Latte', 'Mocha') AND size = 'large'
""")

,drink,size,price
0,Mocha,large,6.5
1,Latte,large,6.0
2,Latte,large,6.0
3,Latte,large,6.0


SQL Date time format: **YYYY-MM-DD HH:MM:SS**

In [53]:
sql("""
SELECT order_id, order_date
FROM orders
WHERE order_date BETWEEN '2026-10-01' AND '2026-10-07'
""")

,order_id,order_date
0,2,2026-10-07
1,10,2026-10-02
2,14,2026-10-06
3,17,2026-10-06
4,21,2026-10-04
5,26,2026-10-05
6,34,2026-10-06
7,35,2026-10-02
8,40,2026-10-01


**Predict:** `%` is any number of characters, `_` is exactly one. Which drinks match each pattern?

In [54]:
# Guess:
sql("""
SELECT DISTINCT drink
FROM orders
WHERE drink LIKE '%Latte'
""")

,drink
0,Chai Latte
1,Latte


In [55]:
# Guess:
sql("""
SELECT DISTINCT drink
FROM orders
WHERE drink LIKE '_a%'
""")

,drink
0,Latte
1,Matcha


**Tinker:** change `'_a%'` to `'%a'`, then `'%e%'`. (In SQLite, `LIKE` ignores upper/lower case. Postgres doesn't.)

**Fix:** this should return Berkeley customers, but it errors. Read the message.

In [56]:
sql("""
SELECT customer_name, city
FROM customers
WHERE city = 'Berkeley'
ORDER BY customer_name
""")

,customer_name,city
0,Ava,Berkeley
1,Chloe,Berkeley
2,Farah,Berkeley
3,Hana,Berkeley


**Your Turn:** using `songs` and `users`. Seven short ones, in order.

1. Every song released in 2020 or later: title, artist, release year. Newest first.

In [57]:
sql("""
SELECT title, artist, release_year
FROM songs
WHERE release_year >= 2020
ORDER BY release_year DESC

""")

,title,artist,release_year
0,Golden Hour Freestyle,Neon Harbor,2025
1,Satellite Hearts,Cinder Avenue,2025
2,Daydream Arcade,Neon Harbor,2025
3,Northbound,MC Parallax,2024
4,Midnight Bus,Rainy Window,2023
5,Coffee Rings,Little Lanterns,2023
6,Heatwave,MC Parallax,2023
7,Glass Skyline,MC Parallax,2022
8,Paper Planes Again,Cinder Avenue,2021
9,Wildfire,Rainy Window,2020


2. Pop or Indie songs (use `IN`) longer than 4 minutes (`duration_sec > 240`).

In [58]:
sql("""
SELECT title, artist, release_year, duration_sec
FROM songs
WHERE genre IN ('Pop', 'Indie') AND duration_sec > 240
""")

,title,artist,release_year,duration_sec
0,Love on Mute,Little Lanterns,2015,248
1,Golden Hour Freestyle,Neon Harbor,2025,286
2,Coffee Rings,Little Lanterns,2023,251
3,Bittersweet,Little Lanterns,2018,277
4,Window Seat,Neon Harbor,2018,261


3. Songs released between 2015 and 2019 (`BETWEEN`), oldest first.

In [59]:
sql("""
SELECT title, artist, release_year, duration_sec
FROM songs
WHERE release_year BETWEEN 2015 and 2019
ORDER BY release_year 
""")

,title,artist,release_year,duration_sec
0,Love on Mute,Little Lanterns,2015,248
1,Love Letters in Pencil,Rainy Window,2017,170
2,Second Wind,Juno Rae,2018,295
3,Bittersweet,Little Lanterns,2018,277
4,Window Seat,Neon Harbor,2018,261
5,Backseat Radio,Little Lanterns,2019,188
6,Overtime,Cinder Avenue,2019,250


4. Every song with **Love** anywhere in the title. (Check: should `Lovesick Algorithm` count?)

In [60]:
sql("""
SELECT title, artist, release_year, duration_sec
FROM songs
WHERE title LIKE '%Love%'
""")

,title,artist,release_year,duration_sec
0,Love on Mute,Little Lanterns,2015,248
1,Static Love,Juno Rae,2010,274
2,Love Letters in Pencil,Rainy Window,2017,170
3,Lovesick Algorithm,Rainy Window,2012,155


5. Usernames that are **exactly 5 characters** long. (Hint: `_` matches exactly one character.)

In [61]:
sql("""
SELECT *
FROM users
WHERE username LIKE '_____'
""")

,user_id,username,country,plan,signup_date,referred_by
0,3,kenji,Canada,free,2025-10-04,1.0
1,4,mango,Canada,free,2025-10-21,NaN
2,6,zara9,US,student,2025-09-26,2.0
3,25,kai_k,UK,student,2025-10-02,NaN


6. The 5 longest songs: title, artist, duration.

In [62]:
sql("""
SELECT title, artist, duration_sec
FROM songs
ORDER BY duration_sec DESC
Limit 5
""")

,title,artist,duration_sec
0,Polaroid,Cinder Avenue,352
1,Paper Planes Again,Cinder Avenue,345
2,Satellite Hearts,Cinder Avenue,334
3,Second Wind,Juno Rae,295
4,Golden Hour Freestyle,Neon Harbor,286


7. Premium users in the US or Canada, earliest signup first.

In [63]:
sql("""
SELECT *
FROM users
WHERE country IN ('US', 'Canada') AND plan = 'premium'
ORDER BY signup_date
""")

,user_id,username,country,plan,signup_date,referred_by
0,18,cadence,US,premium,2025-03-07,4.0
1,13,moonlit_mo,US,premium,2025-04-09,2.0
2,24,harmony_h,Canada,premium,2025-11-21,9.0


---
## 2. Aggregates, GROUP BY, HAVING

**Example:** with no `GROUP BY`, an aggregate squashes the whole table into one row.

In [64]:
sql("""
SELECT COUNT(*) AS num_orders, SUM(price) AS revenue, AVG(price) AS avg_price
FROM orders
""")

,num_orders,revenue,avg_price
0,40,197.25,4.93125


**Predict:** what would this return?

In [65]:
# Guess:
sql("""
SELECT drink, COUNT(*) AS num_orders, AVG(price) AS avg_price
FROM orders
GROUP BY drink
""")

,drink,num_orders,avg_price
0,Chai Latte,8,5.312500
1,Cold Brew,8,4.781250
2,Latte,7,5.428571
3,Matcha,3,5.666667
4,Mocha,6,5.708333
5,Tea,8,3.406250


Same as `orders.groupby("drink")["price"].agg(["count", "mean"])`.

**Example:** group by two columns: one group per unique (drink, size) combo.

In [66]:
sql("""
SELECT drink, size, COUNT(*) AS num_orders
FROM orders
GROUP BY drink, size
ORDER BY drink, size
""")

,drink,size,num_orders
0,Chai Latte,NaN,1
1,Chai Latte,medium,4
2,Chai Latte,small,3
3,Cold Brew,NaN,1
4,Cold Brew,large,1
5,Cold Brew,medium,5
6,Cold Brew,small,1
7,Latte,large,3
8,Latte,medium,2
9,Latte,small,2


Notice the rows where `size` is `None`. `NULL` gets its own group.

**Example:** `HAVING` filters whole groups, after they're counted. Run each cell.

In [67]:
sql("""
SELECT customer_id, COUNT(*) AS num_orders, SUM(price) AS total_spent
FROM orders
GROUP BY customer_id
""")

,customer_id,num_orders,total_spent
0,1,7,31.00
1,2,1,6.00
2,3,11,51.00
3,4,6,32.75
4,5,1,4.75
5,6,7,37.00
6,7,4,21.00
7,9,3,13.75


In [68]:
sql("""
SELECT customer_id, COUNT(*) AS num_orders, SUM(price) AS total_spent
FROM orders
GROUP BY customer_id
HAVING SUM(price) > 30
ORDER BY total_spent DESC
""")

,customer_id,num_orders,total_spent
0,3,11,51.00
1,6,7,37.00
2,4,6,32.75
3,1,7,31.00


**Fix:** this should list customers with more than 4 orders.

In [69]:
sql("""
SELECT customer_id, COUNT(*) AS num_orders
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 4
""")

,customer_id,num_orders
0,1,7
1,3,11
2,4,6
3,6,7


**Your Turn:** aggregates on the music app.

1. How many songs are in each genre? Most songs first. Name the count `num_songs`.

In [70]:
sql("""
SELECT genre, COUNT(*) AS num_songs
FROM songs
GROUP BY genre
ORDER BY num_songs DESC
""")

,genre,num_songs
0,Rock,5
1,R&B,5
2,Pop,5
3,Lo-fi,5
4,Indie,5
5,Hip-Hop,5


2. Average song length per genre **in minutes**. (Hint: `duration_sec / 60.0`. Try `/ 60` too: what goes wrong?)

In [71]:
sql("""
SELECT genre, AVG(duration_sec / 60.0) AS avg_minutes
FROM songs
GROUP BY genre
""")

,genre,avg_minutes
0,Hip-Hop,3.260000
1,Indie,3.910000
2,Lo-fi,2.383333
3,Pop,3.746667
4,R&B,3.816667
5,Rock,5.043333


3. Oldest and newest release year in each genre (`MIN`, `MAX`).

In [72]:
sql("""
SELECT genre, MIN(release_year) AS oldest_year, MAX(release_year) AS newest_year
FROM songs
GROUP BY genre
""")

,genre,oldest_year,newest_year
0,Hip-Hop,2011,2024
1,Indie,2008,2023
2,Lo-fi,2012,2023
3,Pop,2008,2025
4,R&B,2008,2018
5,Rock,2008,2025


4. Number of users for every (country, plan) combo, sorted by country then plan.

In [73]:
sql("""
SELECT country, plan, COUNT(*) AS num_users
FROM users
GROUP BY country, plan
ORDER BY country, plan
""")

,country,plan,num_users
0,Canada,free,2
1,Canada,premium,1
2,Canada,student,2
3,India,free,2
4,India,premium,1
5,India,student,1
6,Mexico,free,3
7,Mexico,premium,1
8,UK,premium,4
9,UK,student,2


5. Total **minutes** listened per `user_id`, highest first.

In [74]:
sql("""
SELECT user_id, SUM(seconds_listened) / 60.0 AS total_minutes
FROM plays
GROUP BY user_id
ORDER BY total_minutes DESC
""")

,user_id,total_minutes
0,20,164.316667
1,21,161.116667
2,10,160.483333
3,25,152.433333
4,19,152.033333
5,5,136.150000
6,3,133.233333
7,1,129.316667
8,9,123.433333
9,24,96.483333


6. Same as 5, but only users who listened for **more than 2 hours**.

In [75]:
sql("""
SELECT user_id, SUM(seconds_listened) / 60.0 AS total_minutes
FROM plays
GROUP BY user_id
HAVING SUM(seconds_listened) / 60.0 > 120
ORDER BY total_minutes DESC
""")

,user_id,total_minutes
0,20,164.316667
1,21,161.116667
2,10,160.483333
3,25,152.433333
4,19,152.033333
5,5,136.150000
6,3,133.233333
7,1,129.316667
8,9,123.433333


7. Genres whose songs average **more than 3.5 minutes**.

In [76]:
sql("""
SELECT genre, AVG(duration_sec / 60.0) AS avg_minutes
FROM songs
GROUP BY genre
HAVING AVG(duration_sec / 60.0) > 3.5
""")

,genre,avg_minutes
0,Indie,3.910000
1,Pop,3.746667
2,R&B,3.816667
3,Rock,5.043333


8. Plays per **month**. `play_date` is text like `'2026-09-14'`, and `SUBSTR(play_date, 1, 7)` keeps the first 7 characters (`'2026-09'`). Group by that.

In [77]:
sql("""
SELECT SUBSTR(play_date, 1, 7) AS month, COUNT(*) AS num_plays
FROM plays
GROUP BY SUBSTR(play_date, 1, 7)
ORDER BY month
""")

,month,num_plays
0,2026-08,238
1,2026-09,248
2,2026-10,237


---
## 3. Joins

`orders` only stores `customer_id`. The names live in `customers`. A join puts them back together.

**Example:** `INNER JOIN` keeps only rows that match in **both** tables.

In [78]:
sql("""
SELECT o.order_id, c.customer_name, o.drink, o.price
FROM orders AS o
INNER JOIN customers AS c ON o.customer_id = c.customer_id
""")

,order_id,customer_name,drink,price
0,2,Farah,Chai Latte,5.50
1,3,Chloe,Chai Latte,5.00
2,4,Farah,Latte,5.25
3,5,Farah,Mocha,5.25
4,6,Chloe,Chai Latte,5.50
5,7,Dev,Cold Brew,4.75
6,8,Farah,Tea,3.50
7,9,Chloe,Mocha,5.25
8,10,Emma,Latte,4.75
9,11,Ava,Cold Brew,4.75


**Predict:** `orders` has 40 rows. How many rows did the inner join return? Why not 40? (Hint: look for a `customer_id` in `orders` that isn't in `customers`.)

In [79]:
# Guess:
sql("""
SELECT COUNT(*) AS rows_after_inner_join
FROM orders AS o
INNER JOIN customers AS c ON o.customer_id = c.customer_id
""")

,rows_after_inner_join
0,37


In [80]:
sql("""
SELECT *
FROM orders
WHERE customer_id NOT IN (SELECT customer_id FROM customers)
""")

,order_id,customer_id,drink,size,price,order_date
0,1,9,Cold Brew,large,5.50,2026-10-18
1,12,9,Tea,medium,3.50,2026-10-19
2,35,9,Cold Brew,medium,4.75,2026-10-02


**Example:** `LEFT JOIN` keeps **every** customer, even ones with no orders. Look for Hana.

In [81]:
sql("""
SELECT c.customer_name, o.drink, o.price
FROM customers AS c
LEFT JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY o.price
""")

,customer_name,drink,price
0,Hana,NaN,NaN
1,Ava,Tea,3.00
2,Chloe,Tea,3.00
3,Chloe,Tea,3.00
4,Ava,Tea,3.50
5,Chloe,Tea,3.50
6,Farah,Tea,3.50
7,Ava,Cold Brew,4.25
8,Dev,Tea,4.25
9,Ava,Cold Brew,4.75


Same as `pd.merge(customers, orders, on="customer_id", how="left")`.

**Example:** join, then group. Total spent per customer, by **name**. Customers who never ordered show up with 0 orders.

In [82]:
sql("""
SELECT c.customer_name, COUNT(o.order_id) AS num_orders, SUM(o.price) AS total_spent
FROM customers AS c
LEFT JOIN orders AS o ON c.customer_id = o.customer_id
GROUP BY c.customer_name
ORDER BY total_spent DESC
""")

,customer_name,num_orders,total_spent
0,Chloe,11,51.00
1,Farah,7,37.00
2,Dev,6,32.75
3,Ava,7,31.00
4,Gabe,4,21.00
5,Ben,1,6.00
6,Emma,1,4.75
7,Hana,0,NaN


**Tinker:** change `COUNT(o.order_id)` to `COUNT(*)`. What happens to Hana's count, and why?

**Example:** `FULL OUTER JOIN` keeps unmatched rows from **both** sides: Hana (no orders) **and** the order from customer 9 (no customer). Skip this cell if your SQLite is older than 3.39.

In [83]:
sql("""
SELECT c.customer_name, o.order_id, o.customer_id
FROM customers AS c
FULL OUTER JOIN orders AS o ON c.customer_id = o.customer_id
WHERE c.customer_id IS NULL OR o.order_id IS NULL
""")

,customer_name,order_id,customer_id
0,Hana,NaN,NaN
1,NaN,1.0,9.0
2,NaN,12.0,9.0
3,NaN,35.0,9.0


**Predict:** this join forgot its `ON`. 8 customers, 40 orders. How many rows come back?

In [84]:
# Guess:
sql("""
SELECT COUNT(*) AS num_rows
FROM customers
JOIN orders
""")

,num_rows
0,320


Every customer paired with every order. That's a cross join: no error, just a huge wrong table.

**Your Turn:** joins. Some of these need **all three** tables: chain a second `JOIN ... ON` right after the first.

1. Each play with the `username` and song `title` next to it. Show 10 rows.

In [85]:
sql("""
SELECT p.play_id, u.username, s.title, p.play_date, p.seconds_listened
FROM plays AS p
INNER JOIN users AS u ON p.user_id = u.user_id
INNER JOIN songs AS s ON p.song_id = s.song_id
LIMIT 10
""")

,play_id,username,title,play_date,seconds_listened
0,1,lofi_luna,City of Lights,2026-08-24,212
1,2,lofi_luna,Lovesick Algorithm,2026-08-25,155
2,3,lofi_luna,Low Tide,2026-09-21,13
3,4,lofi_luna,Wildfire,2026-10-19,104
4,5,lofi_luna,Slow Fade,2026-09-27,9
5,6,lofi_luna,Paper Planes Again,2026-10-16,23
6,7,lofi_luna,Wildfire,2026-09-04,104
7,8,lofi_luna,Paper Planes Again,2026-09-19,345
8,9,lofi_luna,Wildfire,2026-09-27,104
9,10,lofi_luna,City of Lights,2026-09-12,212


2. Number of plays per **username**, including users with zero plays. Fewest first.

In [86]:
sql("""
SELECT u.username, COUNT(p.play_id) AS num_plays
FROM users AS u
LEFT JOIN plays AS p ON u.user_id = p.user_id
GROUP BY u.user_id, u.username
ORDER BY num_plays
""")

,username,num_plays
0,echo,0
1,groove_gus,0
2,noah,6
3,rhythm_raj,7
4,beatsbyben,8
5,cadence,8
6,night_owl,9
7,zara9,16
8,pixel_pia,18
9,tuneup_tia,20


3. Songs that have **never** been played. (Hint: `LEFT JOIN`, then keep rows where the play side `IS NULL`.)

In [87]:
sql("""
SELECT s.song_id, s.title, s.artist
FROM songs AS s
LEFT JOIN plays AS p ON s.song_id = p.song_id
WHERE p.play_id IS NULL
""")

,song_id,title,artist
0,27,Snow Globe,Rainy Window
1,29,Afterglow,Juno Rae


4. Total minutes listened per **genre**, highest first.

In [88]:
sql("""
SELECT s.genre, SUM(p.seconds_listened) / 60.0 AS total_minutes
FROM plays AS p
INNER JOIN songs AS s ON p.song_id = s.song_id
GROUP BY s.genre
ORDER BY total_minutes DESC
""")

,genre,total_minutes
0,Indie,601.533333
1,Rock,481.266667
2,R&B,294.433333
3,Hip-Hop,287.116667
4,Pop,241.966667
5,Lo-fi,148.516667


5. Top 3 **artists** by number of plays.

In [89]:
sql("""
SELECT s.artist, COUNT(*) AS num_plays
FROM plays AS p
INNER JOIN songs AS s ON p.song_id = s.song_id
GROUP BY s.artist
ORDER BY num_plays DESC
LIMIT 3
""")

,artist,num_plays
0,Little Lanterns,180
1,Cinder Avenue,147
2,MC Parallax,142


6. Number of plays from each **country**.

In [90]:
sql("""
SELECT u.country, COUNT(*) AS num_plays
FROM plays AS p
INNER JOIN users AS u ON p.user_id = u.user_id
GROUP BY u.country
ORDER BY num_plays DESC
""")

,country,num_plays
0,Canada,198
1,US,178
2,UK,166
3,India,123
4,Mexico,58


7. Which genre do **premium** users play the most? (Three tables, a `WHERE`, a `GROUP BY`.)

In [91]:
sql("""
SELECT s.genre, COUNT(*) AS num_plays
FROM plays AS p
INNER JOIN users AS u ON p.user_id = u.user_id
INNER JOIN songs AS s ON p.song_id = s.song_id
WHERE u.plan = 'premium'
GROUP BY s.genre
ORDER BY num_plays DESC
""")

,genre,num_plays
0,Rock,56
1,Indie,42
2,Hip-Hop,35
3,R&B,29
4,Pop,24
5,Lo-fi,11


8. **Challenge:** each user next to the username of whoever referred them. `referred_by` holds a `user_id`, so you join `users` to **itself**. Give the two copies different aliases (`u` and `r`). Keep users nobody referred.

In [92]:
sql("""
SELECT u.user_id, u.username, r.username AS referred_by_username
FROM users AS u
LEFT JOIN users AS r ON u.referred_by = r.user_id
""")

,user_id,username,referred_by_username
0,1,lofi_luna,NaN
1,2,beatsbyben,NaN
2,3,kenji,lofi_luna
3,4,mango,NaN
4,5,vinyl_vic,lofi_luna
5,6,zara9,beatsbyben
6,7,night_owl,NaN
7,8,dj_dani,mango
8,9,ritz,lofi_luna
9,10,solar_sam,NaN


---
## 4. CASE, NULLs, Subqueries, CTEs, UNION

**Example:** `CASE` makes a new column with if/else logic. First match wins.

In [93]:
sql("""
SELECT drink, price,
    CASE
        WHEN price >= 6 THEN 'premium'
        WHEN price >= 5 THEN 'regular'
        ELSE 'budget'
    END AS price_tier
FROM orders
LIMIT 10
""")

,drink,price,price_tier
0,Cold Brew,5.50,regular
1,Chai Latte,5.50,regular
2,Chai Latte,5.00,regular
3,Latte,5.25,regular
4,Mocha,5.25,regular
5,Chai Latte,5.50,regular
6,Cold Brew,4.75,budget
7,Tea,3.50,budget
8,Mocha,5.25,regular
9,Latte,4.75,budget


**Tinker:** swap the order of the two `WHEN` lines. Why does nothing come out as `'premium'` anymore?

**Example:** `CASE` inside an aggregate counts things conditionally.

In [94]:
sql("""
SELECT drink,
    COUNT(*) AS num_orders,
    SUM(CASE WHEN size = 'large' THEN 1 ELSE 0 END) AS num_large
FROM orders
GROUP BY drink
""")

,drink,num_orders,num_large
0,Chai Latte,8,0
1,Cold Brew,8,1
2,Latte,7,3
3,Matcha,3,0
4,Mocha,6,1
5,Tea,8,1


**Fix:** this should find orders with no size recorded. It runs, but returns nothing.

In [95]:
sql("""
SELECT order_id, drink, size
FROM orders
WHERE size IS NULL
""")

,order_id,drink,size
0,7,Cold Brew,None
1,8,Tea,None
2,31,Chai Latte,None
3,33,Mocha,None


**Example:** a subquery runs first and hands its answer to the outer query. Run the inner one alone, then the whole thing.

In [96]:
sql("""
SELECT AVG(price) FROM orders
""")

,AVG(price)
0,4.93125


In [97]:
sql("""
SELECT order_id, drink, price
FROM orders
WHERE price > (SELECT AVG(price) FROM orders)
ORDER BY price DESC
""")

,order_id,drink,price
0,17,Mocha,6.50
1,21,Matcha,6.00
2,23,Latte,6.00
3,34,Latte,6.00
4,37,Latte,6.00
5,15,Mocha,5.75
6,27,Mocha,5.75
7,33,Mocha,5.75
8,1,Cold Brew,5.50
9,2,Chai Latte,5.50


**Example:** the same "customers who spent over $30" as before, but now with names: a CTE names the intermediate result, then we use it like a table (like saving a pandas step to a variable).

In [98]:
sql("""
WITH spend AS (
    SELECT customer_id, SUM(price) AS total_spent
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, s.total_spent
FROM spend AS s
INNER JOIN customers AS c ON s.customer_id = c.customer_id
WHERE s.total_spent > 30
ORDER BY s.total_spent DESC
""")

,customer_name,total_spent
0,Chloe,51.00
1,Farah,37.00
2,Dev,32.75
3,Ava,31.00


**Predict:** `UNION` removes duplicate rows, `UNION ALL` keeps them. How many rows will each return?

In [99]:
# Guess: UNION ___ rows, UNION ALL ___ rows
sql("""
SELECT city FROM customers WHERE customer_id <= 3
UNION
SELECT city FROM customers WHERE customer_id >= 6
""")

,city
0,Berkeley
1,Oakland
2,San Francisco


In [100]:
sql("""
SELECT city FROM customers WHERE customer_id <= 3
UNION ALL
SELECT city FROM customers WHERE customer_id >= 6
""")

,city
0,Berkeley
1,Oakland
2,Berkeley
3,Berkeley
4,San Francisco
5,Berkeley


**Your Turn:** CASE, NULLs, subqueries, CTEs, UNION.

1. Label every song `'short'` (under 2.5 min), `'medium'` (2.5 to 4 min), or `'long'` (over 4 min). Show title, duration, and the label.

In [101]:
sql("""
SELECT title, duration_sec,
    CASE
        WHEN duration_sec < 150 THEN 'short'
        WHEN duration_sec <= 240 THEN 'medium'
        ELSE 'long'
    END AS length_label
FROM songs
""")

,title,duration_sec,length_label
0,Love on Mute,248,long
1,Glass Skyline,140,short
2,Midnight Bus,154,medium
3,Paper Planes Again,345,long
4,Static Love,274,long
5,Golden Hour Freestyle,286,long
6,Coffee Rings,251,long
7,Northbound,258,long
8,Love Letters in Pencil,170,medium
9,Satellite Hearts,334,long


2. Count how many songs got each label from Q1. (Hint: put the whole `CASE ... END` in the `GROUP BY` too, or group by the alias.)

In [102]:
sql("""
SELECT
    CASE
        WHEN duration_sec < 150 THEN 'short'
        WHEN duration_sec <= 240 THEN 'medium'
        ELSE 'long'
    END AS length_label,
    COUNT(*) AS num_songs
FROM songs
GROUP BY length_label
""")

,length_label,num_songs
0,long,13
1,medium,14
2,short,3


3. A play under 30 seconds is a **skip**. For each `user_id`, count total plays and skips (`SUM(CASE ...)`).

In [103]:
sql("""
SELECT user_id,
    COUNT(*) AS num_plays,
    SUM(CASE WHEN seconds_listened < 30 THEN 1 ELSE 0 END) AS num_skips
FROM plays
GROUP BY user_id
""")

,user_id,num_plays,num_skips
0,1,51,13
1,2,8,2
2,3,42,8
3,4,23,1
4,5,52,15
5,6,16,4
6,7,9,3
7,8,35,15
8,9,55,17
9,10,55,14


4. Users nobody referred have `NULL` in `referred_by`. List those users with `IS NULL`.

In [104]:
sql("""
SELECT user_id, username
FROM users
WHERE referred_by IS NULL
""")

,user_id,username
0,1,lofi_luna
1,2,beatsbyben
2,4,mango
3,7,night_owl
4,10,solar_sam
5,12,echo
6,14,tuneup_tia
7,17,bassline_bo
8,19,indie_ines
9,21,melody_mia


5. Songs **longer than the average** song (subquery).

In [105]:
sql("""
SELECT title, artist, duration_sec
FROM songs
WHERE duration_sec > (SELECT AVG(duration_sec) FROM songs)
ORDER BY duration_sec DESC
""")

,title,artist,duration_sec
0,Polaroid,Cinder Avenue,352
1,Paper Planes Again,Cinder Avenue,345
2,Satellite Hearts,Cinder Avenue,334
3,Second Wind,Juno Rae,295
4,Golden Hour Freestyle,Neon Harbor,286
5,Bittersweet,Little Lanterns,277
6,Static Love,Juno Rae,274
7,Window Seat,Neon Harbor,261
8,Northbound,MC Parallax,258
9,Coffee Rings,Little Lanterns,251


6. Users who **never** played anything (`NOT IN` subquery). Compare with your answer to Joins Q2.

In [106]:
sql("""
SELECT user_id, username
FROM users
WHERE user_id NOT IN (SELECT user_id FROM plays)
""")

,user_id,username
0,12,echo
1,23,groove_gus


7. Songs from the **most-played artist**. (The subquery finds the artist; the outer query lists their songs.)

In [107]:
sql("""
SELECT title, artist
FROM songs
WHERE artist = (
    SELECT s.artist
    FROM plays AS p
    INNER JOIN songs AS s ON p.song_id = s.song_id
    GROUP BY s.artist
    ORDER BY COUNT(*) DESC
    LIMIT 1
)
""")

,title,artist
0,Love on Mute,Little Lanterns
1,Coffee Rings,Little Lanterns
2,Fever Dream,Little Lanterns
3,Backseat Radio,Little Lanterns
4,Bittersweet,Little Lanterns


8. With a CTE: each user's total minutes, then the usernames of everyone **above the average** user.

In [108]:
sql("""
WITH user_minutes AS (
    SELECT user_id, SUM(seconds_listened) / 60.0 AS total_minutes
    FROM plays
    GROUP BY user_id
)
SELECT u.username, um.total_minutes
FROM user_minutes AS um
INNER JOIN users AS u ON um.user_id = u.user_id
WHERE um.total_minutes > (SELECT AVG(total_minutes) FROM user_minutes)
ORDER BY um.total_minutes DESC
""")

,username,total_minutes
0,loop_leo,164.316667
1,melody_mia,161.116667
2,solar_sam,160.483333
3,kai_k,152.433333
4,indie_ines,152.033333
5,vinyl_vic,136.150000
6,kenji,133.233333
7,lofi_luna,129.316667
8,ritz,123.433333
9,harmony_h,96.483333


9. One column of every **name** in the app: all usernames and all artist names. Add a second column saying `'user'` or `'artist'`. Use `UNION`. Then try `UNION ALL`: why does the row count jump?

In [109]:
# UNION ALL keeps duplicates: every artist appears once per song they have, so the row count jumps.
sql("""
SELECT username AS name, 'user' AS type FROM users
UNION
SELECT artist AS name, 'artist' AS type FROM songs
""")

,name,type
0,Cinder Avenue,artist
1,Juno Rae,artist
2,Little Lanterns,artist
3,MC Parallax,artist
4,Neon Harbor,artist
5,Rainy Window,artist
6,aria,user
7,bassline_bo,user
8,beatsbyben,user
9,cadence,user


---
## 5. Saving Results: CREATE TABLE and CREATE VIEW

`sql()` only reads. To change the database, use `conn.execute`.

**Example:** `CREATE TABLE ... AS` saves a query's result as a real table (a snapshot). `CREATE VIEW` saves the **query itself** and reruns it every time you use it.

In [110]:
conn.execute("DROP TABLE IF EXISTS spend_snapshot")
conn.execute("DROP VIEW IF EXISTS spend_view")

conn.execute("""
CREATE TABLE spend_snapshot AS
SELECT customer_id, SUM(price) AS total_spent FROM orders GROUP BY customer_id
""")
conn.execute("""
CREATE VIEW spend_view AS
SELECT customer_id, SUM(price) AS total_spent FROM orders GROUP BY customer_id
""")
sql("SELECT name, type FROM sqlite_master WHERE name LIKE 'spend%'")

,name,type
0,spend_snapshot,table
1,spend_view,view


**Predict:** we add a $100 order for customer 1. Which one updates, the snapshot or the view?

In [111]:
# Guess:
conn.execute("INSERT INTO orders VALUES (41, 1, 'Latte', 'large', 100.0, '2026-10-31')")

print(sql("SELECT * FROM spend_snapshot WHERE customer_id = 1"))
print(sql("SELECT * FROM spend_view WHERE customer_id = 1"))

   customer_id  total_spent
0            1         31.0
   customer_id  total_spent
0            1        131.0


A **materialized view** (Postgres) is in between: it saves a snapshot like a table, but remembers its query so you can refresh it. SQLite doesn't have them.

Undo the fake order so reruns stay clean:

In [112]:
conn.execute("DELETE FROM orders WHERE order_id = 41")
conn.commit()

**Your Turn:**

1. Create a view `song_stats`: each song's title, artist, number of plays, and total minutes listened (unplayed songs included, with 0 plays). Then query it for the 5 songs with the most minutes.

In [113]:
conn.execute("DROP VIEW IF EXISTS song_stats")
conn.execute("""
CREATE VIEW song_stats AS
SELECT s.song_id, s.title, s.artist,
    COUNT(p.play_id) AS num_plays,
    COALESCE(SUM(p.seconds_listened), 0) / 60.0 AS total_minutes
FROM songs AS s
LEFT JOIN plays AS p ON s.song_id = p.song_id
GROUP BY s.song_id, s.title, s.artist
""")
sql("""
SELECT title, artist, total_minutes
FROM song_stats
ORDER BY total_minutes DESC
LIMIT 5
""")

,title,artist,total_minutes
0,Bittersweet,Little Lanterns,220.116667
1,Paper Planes Again,Cinder Avenue,165.833333
2,Second Wind,Juno Rae,159.166667
3,Coffee Rings,Little Lanterns,145.233333
4,Love on Mute,Little Lanterns,135.083333


2. Use your view (no joins needed now): which songs have **0 plays**?

In [114]:
sql("""
SELECT song_id, title, artist
FROM song_stats
WHERE num_plays = 0
""")

,song_id,title,artist
0,27,Snow Globe,Rainy Window
1,29,Afterglow,Juno Rae


3. Save a **snapshot** table `song_stats_snapshot` from the view (`CREATE TABLE ... AS SELECT * FROM song_stats`). Then insert one play of song 27 by user 1 and check song 27 in both. Delete the play afterwards.

In [115]:
conn.execute("DROP TABLE IF EXISTS song_stats_snapshot")
conn.execute("CREATE TABLE song_stats_snapshot AS SELECT * FROM song_stats")
conn.execute("INSERT INTO plays VALUES (99999, 1, 27, '2026-10-31', 200)")

print(sql("SELECT * FROM song_stats_snapshot WHERE song_id = 27"))
print(sql("SELECT * FROM song_stats WHERE song_id = 27"))

conn.execute("DELETE FROM plays WHERE play_id = 99999")
conn.commit()

   song_id       title        artist  num_plays  total_minutes
0       27  Snow Globe  Rainy Window          0            0.0
   song_id       title        artist  num_plays  total_minutes
0       27  Snow Globe  Rainy Window          1       3.333333


---
## 6. Challenge Round

These are interview-style: each one takes a few steps. Plan it in plain English first (*which tables? join? group by what? filter before or after grouping?*), then write the SQL. Use a CTE whenever a step needs its own name.

Two new tools you may want:
- `COUNT(DISTINCT column)` counts unique values instead of rows.
- `ROUND(x, 2)` rounds to 2 decimals.

**C1.** Skip rate per genre: the fraction of plays under 30 seconds. Highest skip rate first. (Hint: `AVG(CASE WHEN ... THEN 1.0 ELSE 0 END)`.)

In [116]:
sql("""
SELECT s.genre,
    ROUND(AVG(CASE WHEN p.seconds_listened < 30 THEN 1.0 ELSE 0 END), 2) AS skip_rate
FROM plays AS p
INNER JOIN songs AS s ON p.song_id = s.song_id
GROUP BY s.genre
ORDER BY skip_rate DESC
""")

,genre,skip_rate
0,Rock,0.38
1,Pop,0.32
2,Hip-Hop,0.31
3,R&B,0.22
4,Indie,0.20
5,Lo-fi,0.14


**C2.** Which 5 users played the most **different songs**? Show their total plays too. Who replays the same songs the most?

In [117]:
sql("""
SELECT u.username,
    COUNT(DISTINCT p.song_id) AS distinct_songs,
    COUNT(*) AS total_plays,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT p.song_id), 2) AS plays_per_song
FROM plays AS p
INNER JOIN users AS u ON p.user_id = u.user_id
GROUP BY u.user_id, u.username
ORDER BY distinct_songs DESC
LIMIT 5
""")

,username,distinct_songs,total_plays,plays_per_song
0,solar_sam,22,55,2.50
1,melody_mia,21,59,2.81
2,lofi_luna,20,51,2.55
3,ritz,19,55,2.89
4,vinyl_vic,18,52,2.89


**C3.** Do premium users listen more? Average minutes **per user** for each plan. Careful: first total each user, *then* average those totals by plan (CTE). Include users with 0 minutes.

In [118]:
sql("""
WITH user_minutes AS (
    SELECT u.user_id, u.plan, COALESCE(SUM(p.seconds_listened), 0) / 60.0 AS total_minutes
    FROM users AS u
    LEFT JOIN plays AS p ON u.user_id = p.user_id
    GROUP BY u.user_id, u.plan
)
SELECT plan, COUNT(*) AS num_users, ROUND(AVG(total_minutes), 2) AS avg_minutes_per_user
FROM user_minutes
GROUP BY plan
ORDER BY avg_minutes_per_user DESC
""")

,plan,num_users,avg_minutes_per_user
0,student,8,96.33
1,free,8,89.42
2,premium,9,63.21


**C4.** Users who referred **at least 2** people. Show the referrer's username and how many they referred.

In [119]:
sql("""
SELECT r.username AS referrer, COUNT(*) AS num_referred
FROM users AS u
INNER JOIN users AS r ON u.referred_by = r.user_id
GROUP BY r.user_id, r.username
HAVING COUNT(*) >= 2
ORDER BY num_referred DESC
""")

,referrer,num_referred
0,lofi_luna,3
1,mango,3
2,beatsbyben,2
3,ritz,2


**C5.** Did referred users listen more than non-referred users? Use `CASE` to label each user `'referred'` or `'organic'`, then compare average total minutes per user.

In [120]:
sql("""
WITH user_minutes AS (
    SELECT u.user_id,
        CASE WHEN u.referred_by IS NULL THEN 'organic' ELSE 'referred' END AS source,
        COALESCE(SUM(p.seconds_listened), 0) / 60.0 AS total_minutes
    FROM users AS u
    LEFT JOIN plays AS p ON u.user_id = p.user_id
    GROUP BY u.user_id, source
)
SELECT source, COUNT(*) AS num_users, ROUND(AVG(total_minutes), 2) AS avg_minutes_per_user
FROM user_minutes
GROUP BY source
""")

,source,num_users,avg_minutes_per_user
0,organic,12,82.41
1,referred,13,82.00


**C6.** Each song's play count compared with its **genre's** average play count. Show only songs that beat their genre's average. (Hint: CTE 1 = plays per song. CTE 2 = average of CTE 1 per genre. Join them on genre.)

In [121]:
sql("""
WITH song_plays AS (
    SELECT s.song_id, s.title, s.genre, COUNT(p.play_id) AS num_plays
    FROM songs AS s
    LEFT JOIN plays AS p ON s.song_id = p.song_id
    GROUP BY s.song_id, s.title, s.genre
),
genre_avg AS (
    SELECT genre, AVG(num_plays) AS avg_plays
    FROM song_plays
    GROUP BY genre
)
SELECT sp.title, sp.genre, sp.num_plays, ROUND(ga.avg_plays, 2) AS genre_avg_plays
FROM song_plays AS sp
INNER JOIN genre_avg AS ga ON sp.genre = ga.genre
WHERE sp.num_plays > ga.avg_plays
ORDER BY sp.genre, sp.num_plays DESC
""")

,title,genre,num_plays,genre_avg_plays
0,Glass Skyline,Hip-Hop,77,28.4
1,Bittersweet,Indie,58,36.0
2,Coffee Rings,Indie,45,36.0
3,Love on Mute,Indie,40,36.0
4,Lovesick Algorithm,Lo-fi,29,14.8
5,Wildfire,Lo-fi,27,14.8
6,Cloud Nine Remix,Pop,40,19.4
7,Window Seat,Pop,28,19.4
8,Second Wind,R&B,39,16.6
9,Static Love,R&B,21,16.6


**C7.** The **most-played genre in each country**. (Hint: CTE 1 = plays per (country, genre). CTE 2 = the max of that per country. Join them where the count equals the max. Ties can give two rows, and that's fine.)

In [122]:
sql("""
WITH country_genre AS (
    SELECT u.country, s.genre, COUNT(*) AS num_plays
    FROM plays AS p
    INNER JOIN users AS u ON p.user_id = u.user_id
    INNER JOIN songs AS s ON p.song_id = s.song_id
    GROUP BY u.country, s.genre
),
country_max AS (
    SELECT country, MAX(num_plays) AS max_plays
    FROM country_genre
    GROUP BY country
)
SELECT cg.country, cg.genre, cg.num_plays
FROM country_genre AS cg
INNER JOIN country_max AS cm
    ON cg.country = cm.country AND cg.num_plays = cm.max_plays
ORDER BY cg.country
""")

,country,genre,num_plays
0,Canada,Indie,111
1,India,Lo-fi,40
2,Mexico,Pop,29
3,UK,Rock,82
4,US,Hip-Hop,80


---
## 7. Translate pandas to SQL

You already know pandas. Each cell shows a pandas line on the music data; write the SQL that gives the same table. Run both and compare. (The DataFrames `users`, `songs`, `plays` from the setup cell are still in memory.)

**P1.** Filter and sort.

In [123]:
songs[songs["genre"] == "Hip-Hop"].sort_values("release_year")[["title", "release_year"]]

,title,release_year
19,Low Tide,2011
13,Late Checkout,2012
1,Glass Skyline,2022
25,Heatwave,2023
7,Northbound,2024


In [124]:
sql("""
SELECT title, release_year
FROM songs
WHERE genre = 'Hip-Hop'
ORDER BY release_year
""")

,title,release_year
0,Low Tide,2011
1,Late Checkout,2012
2,Glass Skyline,2022
3,Heatwave,2023
4,Northbound,2024


**P2.** Group and count.

In [125]:
users.groupby("plan")["user_id"].count()

plan
free       8
premium    9
student    8
Name: user_id, dtype: int64

In [126]:
sql("""
SELECT plan, COUNT(user_id) AS user_id
FROM users
GROUP BY plan
""")

,plan,user_id
0,free,8
1,premium,9
2,student,8


**P3.** Merge, then group.

In [127]:
pd.merge(plays, songs, on="song_id").groupby("artist")["seconds_listened"].sum().sort_values(ascending=False)

artist
Little Lanterns    36092
Cinder Avenue      28876
Juno Rae           17666
MC Parallax        17227
Neon Harbor        14518
Rainy Window        8911
Name: seconds_listened, dtype: int64

In [128]:
sql("""
SELECT s.artist, SUM(p.seconds_listened) AS seconds_listened
FROM plays AS p
INNER JOIN songs AS s ON p.song_id = s.song_id
GROUP BY s.artist
ORDER BY seconds_listened DESC
""")

,artist,seconds_listened
0,Little Lanterns,36092
1,Cinder Avenue,28876
2,Juno Rae,17666
3,MC Parallax,17227
4,Neon Harbor,14518
5,Rainy Window,8911


**P4.** Left merge and find missing matches.

In [129]:
m = pd.merge(users, plays, on="user_id", how="left")
m[m["play_id"].isna()][["username"]]

,username
364,echo
643,groove_gus


In [130]:
sql("""
SELECT u.username
FROM users AS u
LEFT JOIN plays AS p ON u.user_id = p.user_id
WHERE p.play_id IS NULL
""")

,username
0,echo
1,groove_gus


---
## Done: Push Your Work

Run these in your terminal (not in this notebook). Don't commit `week7.db`; the notebook rebuilds it.

```
git add Week7.ipynb
git commit -m "week 7 notebook"
git push origin YOUR-BRANCH-NAME
```